In [13]:
import pandas as pd
import sqlite3

df = pd.read_csv('/Users/akashkumarsamantray/food-delivery-germany/food-delivery-market-analysis-germany/data/cleaned_reviews.csv')
df['at'] = pd.to_datetime(df['at'])
df


,reviewId,userName,content,score,thumbsUpCount,at,replyContent,repliedAt,appVersion,app_name,review_year,review_month,review_day,review_hour,sentiment,review_length,word_count,has_reply
0,06626e61-d64f-4e6c-933e-d581256aac7b,Sara Levermann,Ich finde diese App sehr gut.🙂...,5,0,2026-06-09 21:59:48,No Reply,NaN,26.23.3,Wolt,2026,6,Tuesday,21,Positive,33,6,0
1,97aad388-9203-45eb-8499-db5de7b7e666,Matthias Ri,If anything goes wrong with an order don't exp...,1,0,2026-06-09 06:17:59,No Reply,NaN,26.22.3,Wolt,2026,6,Tuesday,6,Negative,107,17,0
2,b96870a4-171e-48cc-bf40-3e4140b649cc,Jean Wolff,Menü für 2 Personnen bestellt und gezahlt. Ein...,1,1,2026-06-08 11:58:17,No Reply,NaN,26.22.3,Wolt,2026,6,Monday,11,Negative,409,60,0
3,ec2966b1-2f78-4920-b4be-fb613c8b0689,Latifa Khennoussi,tolle App,5,0,2026-06-07 23:16:23,No Reply,NaN,26.22.3,Wolt,2026,6,Sunday,23,Positive,9,2,0
4,97cf249e-188c-4c95-88f6-0eef32b63e24,Wahid Alisaleha,Super,5,0,2026-06-07 21:40:12,No Reply,NaN,26.22.3,Wolt,2026,6,Sunday,21,Positive,5,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5994,ef96ec8e-5ea9-423d-95a6-90bb0f65c510,Fadi Abdul Mouti,Benutzt einen anderen Lieferservice vertraut e...,1,1,2025-04-24 15:38:59,"Hallo! Es tut uns leid zu hören, dass es Probl...",2025-06-06 18:50:39,6.267.10000,Uber Eats,2025,4,Thursday,15,Negative,89,10,1
5995,64708076-d55e-4b5d-baf1-fc01e5962498,Kage no Hana,app deinstalliert! schrecklich! hatte irgendwi...,1,1,2025-04-23 20:56:46,"Hallo!\nEs tut uns leid zu hören, dass es Prob...",2025-06-06 18:51:45,Unknown,Uber Eats,2025,4,Wednesday,20,Negative,432,58,1
5996,9577c863-9be7-461b-b4f4-b65a4c56c737,Anton Andre Mecklenburg-B,"Viel zu wenig Restaurants, wie z. Beispiel ein...",3,0,2025-04-23 18:49:29,No Reply,NaN,Unknown,Uber Eats,2025,4,Wednesday,18,Neutral,71,10,0
5997,bd0bfb7c-cb46-4314-a99e-f7f64311ed28,Markus E.,verseucht mit Bannern und irreführenden Button...,1,0,2025-04-23 18:35:43,No Reply,NaN,6.267.10000,Uber Eats,2025,4,Wednesday,18,Negative,92,12,0


In [14]:
## Created SQLite database

conn = sqlite3.connect('data/food_delivery.db')
df.to_sql('reviews', conn, if_exists='replace', index=False)
print("Database created successfully!")

Database created successfully!


In [15]:
## Ran a Helper Function

def run_query(query):
    return pd.read_sql_query(query, conn)

In [16]:
## Q1: Average score per app

run_query("""
    SELECT app_name,
           ROUND(AVG(score), 2) AS avg_score,
           COUNT(*) AS total_reviews
    FROM reviews
    GROUP BY app_name
    ORDER BY avg_score DESC
""")

,app_name,avg_score,total_reviews
0,Wolt,3.43,2000
1,Lieferando,2.97,2000
2,Uber Eats,1.70,1999


In [17]:
## Q2: Sentiment breakdown per app

run_query("""
    SELECT app_name,
           sentiment,
           COUNT(*) AS count,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY app_name), 2) AS percentage
    FROM reviews
    GROUP BY app_name, sentiment
    ORDER BY app_name, sentiment
""")

,app_name,sentiment,count,percentage
0,Lieferando,Negative,973,48.65
1,Lieferando,Neutral,97,4.85
2,Lieferando,Positive,930,46.50
3,Uber Eats,Negative,1634,81.74
4,Uber Eats,Neutral,68,3.40
5,Uber Eats,Positive,297,14.86
6,Wolt,Negative,760,38.00
7,Wolt,Neutral,41,2.05
8,Wolt,Positive,1199,59.95


In [18]:
## Q3: Monthly review trend

run_query("""
    SELECT app_name,
           review_year,
           review_month,
           COUNT(*) AS total_reviews,
           ROUND(AVG(score), 2) AS avg_score
    FROM reviews
    GROUP BY app_name, review_year, review_month
    ORDER BY review_year, review_month
""")

,app_name,review_year,review_month,total_reviews,avg_score
0,Wolt,2024,5,4,3.00
1,Wolt,2024,6,55,2.84
2,Wolt,2024,7,62,3.48
3,Wolt,2024,8,59,3.44
4,Wolt,2024,9,71,3.38
5,Wolt,2024,10,65,3.32
6,Wolt,2024,11,96,3.18
7,Wolt,2024,12,70,3.39
8,Wolt,2025,1,82,3.46
9,Wolt,2025,2,74,3.38


In [19]:
## Q4: App response rate (customer service)

run_query("""
    SELECT app_name,
           COUNT(*) AS total_reviews,
           SUM(has_reply) AS replied,
           ROUND(SUM(has_reply) * 100.0 / COUNT(*), 2) AS reply_rate_pct
    FROM reviews
    GROUP BY app_name
    ORDER BY reply_rate_pct DESC
""")

,app_name,total_reviews,replied,reply_rate_pct
0,Lieferando,2000,1254,62.70
1,Uber Eats,1999,463,23.16
2,Wolt,2000,0,0.00


In [20]:
## Q5: Most thumbed-up negative reviews (biggest pain points)

run_query("""
    SELECT app_name,
           content,
           score,
           thumbsUpCount
    FROM reviews
    WHERE sentiment = 'Negative'
    ORDER BY thumbsUpCount DESC
    LIMIT 10
""")

,app_name,content,score,thumbsUpCount
0,Uber Eats,Mir geht es leider wie vielen hier. Mir wurde ...,1,249
1,Uber Eats,Leider erst jetzt die Bewertungen hier gelesen...,1,232
2,Wolt,"Die Berechnung von zig Liefer-, Zusatzgebühren...",1,212
3,Uber Eats,Grenzt an Betrug. Nach Einlösen eines Gutschei...,1,203
4,Uber Eats,"Ein Rabattcode wurde mir in der App angezeigt,...",1,195
5,Wolt,Ich habe eine falsche Bestellung erhalten und ...,2,181
6,Uber Eats,Betrüger!! Haben mit einem 15€ Gutschein gewor...,1,156
7,Uber Eats,"Die App ist grottenschlecht, nichts funktionie...",1,145
8,Uber Eats,Preislich top Angebote (regelmäßig 2 für 1 Ang...,2,124
9,Uber Eats,Die App funktioniert. Nur das Angebot wird imm...,1,122


In [ ]:
## SQL INSIGHTS

In [ ]:
## Reply Rate (Customer Service)

## Lieferando replies to 62.70% of reviews — most responsive.
## Uber Eats only 23.16% — poor engagement.
## Wolt 0% — never replies publicly (but has best ratings!).

In [ ]:
## Most Thumbed-Up Complaints (Top Pain Points)

## Uber Eats dominates the top complaints — voucher fraud (Gutschein), app failures, and money issues.
## Wolt's top complaint — hidden delivery fees (212 thumbs up) and wrong orders (181 thumbs up).
## The word "Betrüger" (fraudster/scammer) appears in Uber Eats reviews — very damaging.

In [ ]:
## Monthly Trend — Big Discovery!

## Lieferando had near-zero reviews until Jan 2026, then suddenly exploded to 515 in Feb 2026 — 
## something major happened (app update, media story, or price increase).

In [10]:
## Exported SQL results for Tableau

queries = {
    'avg_score': """
        SELECT app_name, ROUND(AVG(score),2) AS avg_score, COUNT(*) AS total_reviews
        FROM reviews GROUP BY app_name
    """,
    'sentiment_breakdown': """
        SELECT app_name, sentiment, COUNT(*) AS count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY app_name), 2) AS percentage
        FROM reviews GROUP BY app_name, sentiment
    """,
    'monthly_trend': """
        SELECT app_name, review_year, review_month,
        COUNT(*) AS total_reviews, ROUND(AVG(score),2) AS avg_score
        FROM reviews GROUP BY app_name, review_year, review_month
        ORDER BY review_year, review_month
    """,
    'reply_rate': """
        SELECT app_name, COUNT(*) AS total_reviews,
        SUM(has_reply) AS replied,
        ROUND(SUM(has_reply) * 100.0 / COUNT(*), 2) AS reply_rate_pct
        FROM reviews GROUP BY app_name
    """
}

for name, query in queries.items():
    result = pd.read_sql_query(query, conn)
    result.to_csv(f'data/tableau_{name}.csv', index=False)
    print(f"Saved tableau_{name}.csv ✅")

conn.close()
print("\nAll Tableau exports done!")

Saved tableau_avg_score.csv ✅
Saved tableau_sentiment_breakdown.csv ✅
Saved tableau_monthly_trend.csv ✅
Saved tableau_reply_rate.csv ✅

All Tableau exports done!


In [11]:
import os
print(os.listdir('data'))

['tableau_sentiment_breakdown.csv', 'raw_reviews.csv', 'tableau_monthly_trend.csv', 'summary_stats.csv', 'tableau_reply_rate.csv', 'food_delivery.db', 'cleaned_reviews.csv', 'tableau_avg_score.csv']


In [12]:
trend = pd.read_csv('/Users/akashkumarsamantray/food-delivery-germany/food-delivery-market-analysis-germany/data/tableau_monthly_trend.csv')
trend[trend['app_name'] == 'Lieferando']

,app_name,review_year,review_month,total_reviews,avg_score
29,Lieferando,2026,1,31,2.81
32,Lieferando,2026,2,515,2.86
35,Lieferando,2026,3,488,3.04
38,Lieferando,2026,4,336,2.75
41,Lieferando,2026,5,481,3.13
44,Lieferando,2026,6,149,3.11
